# Fine-tune Kinyarwanda TTS (MMS-VITS) on the podcast host — Kaggle, free GPU

**Before running:** Settings → Accelerator → **GPU T4 x2** (or P100), Internet → **On**.
Add Input → your **private** dataset `kin-host` (the zip from `uv run kintts-export`).

We do **not** start from scratch: we continue training Meta's `facebook/mms-tts-kin`,
which already speaks Kinyarwanda, so it only has to learn the host's voice and rhythm.

To continue in a new session (Kaggle stops after 12 h): *Add Input → Notebook output* of the
previous run, then set `RESUME_FROM` below. Your data stays private in your Kaggle account;
delete the dataset after the demo if required.

In [ ]:
RESUME_FROM = None          # e.g. "/kaggle/input/kin-tts-finetune/vits_kin_host"
DATA_DIR    = "/kaggle/input/kin-host"
OUT_DIR     = "/kaggle/working/vits_kin_host"
EPOCHS      = 150           # ~6-10 h on T4 for ~5 h of audio; lower for a quick test
SAMPLE_TEXT = "muraho neza, uyu munsi tuzaganira ku mateka y'u rwanda n'uko igihugu cyacu cyiyubatse"


In [ ]:
%%bash
set -e
cd /kaggle/working
[ -d finetune-hf-vits ] || git clone -q https://github.com/ylacombe/finetune-hf-vits
pip install -q "transformers==4.46.3" "datasets[audio]==2.21.0" "accelerate==1.0.1" Cython tensorboard matplotlib
cd finetune-hf-vits/monotonic_align && mkdir -p monotonic_align && python setup.py build_ext --inplace > /dev/null && echo "monotonic_align built"


In [ ]:
import os, glob, shutil
# Kaggle auto-extracts the zip; find the folder that contains train/metadata.csv
meta = glob.glob(f"{DATA_DIR}/**/train/metadata.csv", recursive=True)[0]
dataset_root = os.path.dirname(os.path.dirname(meta))
print("dataset:", dataset_root, "clips:", sum(1 for _ in open(meta)) - 1)

# Starting point: MMS Kinyarwanda generator + its original discriminator (needed for GAN training)
BASE = "/kaggle/working/mms-tts-kin-train"
if not os.path.exists(BASE):
    !cd /kaggle/working/finetune-hf-vits && python convert_original_discriminator_checkpoint.py --language_code kin --pytorch_dump_folder_path {BASE}
if RESUME_FROM and not os.path.exists(OUT_DIR):
    shutil.copytree(RESUME_FROM, OUT_DIR)


In [ ]:
import json
config = {
    "project_name": "kin_host",
    "push_to_hub": False,
    "report_to": ["tensorboard"],
    "overwrite_output_dir": False,
    "output_dir": OUT_DIR,
    "resume_from_checkpoint": "latest" if RESUME_FROM else None,

    "dataset_name": dataset_root,
    "audio_column_name": "audio",
    "text_column_name": "text",
    "train_split_name": "train",
    "eval_split_name": "train",
    "full_generation_sample_text": SAMPLE_TEXT,

    "max_duration_in_seconds": 15,
    "min_duration_in_seconds": 1.0,
    "max_tokens_length": 500,
    "do_lower_case": True,

    "model_name_or_path": BASE,
    "preprocessing_num_workers": 4,

    "do_train": True,
    "num_train_epochs": EPOCHS,
    "gradient_accumulation_steps": 1,
    "per_device_train_batch_size": 16,
    "learning_rate": 2e-5,
    "adam_beta1": 0.8,
    "adam_beta2": 0.99,
    "warmup_ratio": 0.01,
    "group_by_length": False,
    "save_steps": 1000,

    "do_eval": True,
    "eval_steps": 500,
    "per_device_eval_batch_size": 16,
    "max_eval_samples": 16,
    "do_step_schedule_per_epoch": True,

    "weight_disc": 3, "weight_fmaps": 1, "weight_gen": 1,
    "weight_kl": 1.5, "weight_duration": 1, "weight_mel": 35,

    "fp16": True,
    "seed": 456,
}
config = {k: v for k, v in config.items() if v is not None}
json.dump(config, open("/kaggle/working/finetune_kin.json", "w"), indent=2)


In [ ]:
!cd /kaggle/working/finetune-hf-vits && accelerate launch --num_processes 1 run_vits_finetuning.py /kaggle/working/finetune_kin.json

## Listen to the result

In [ ]:
import torch, IPython.display as ipd
from transformers import VitsModel, AutoTokenizer
m = VitsModel.from_pretrained(OUT_DIR).eval(); t = AutoTokenizer.from_pretrained(OUT_DIR)
for s in [SAMPLE_TEXT, "ese wowe ukunda kurya iki?", "murakoze cyane ku bufasha bwanyu"]:
    with torch.no_grad(): w = m(**t(s, return_tensors="pt")).waveform[0].numpy()
    ipd.display(ipd.Audio(w, rate=m.config.sampling_rate))


## Take the model home
Zip only the inference files (small, ~150 MB). Download `kin_host_model.zip` from the Output panel,
unzip it into `kinyarwanda-tts/checkpoints/host-v1/`, then run `uv run kintts-app` — it appears in the Model list.

In [ ]:
import shutil, os
export = "/kaggle/working/kin_host_model"
os.makedirs(export, exist_ok=True)
for f in os.listdir(OUT_DIR):
    if f.endswith((".json", ".safetensors", ".bin", ".txt")) and os.path.isfile(f"{OUT_DIR}/{f}"):
        shutil.copy(f"{OUT_DIR}/{f}", export)
shutil.make_archive("/kaggle/working/kin_host_model", "zip", export)
print(os.listdir(export))
